# Notebook 03 — Model Architecture

## Building a Foundation Model from Scratch

This notebook defines, implements, and validates the decoder-only Transformer architecture used in the controlled model-scaling experiment.

### Experimental architecture family

We will implement three progressively larger members of the same architectural family while holding the tokenizer, dataset, context length, training objective, and training methodology constant.

| Model | Layers | d_model | Heads | Head Dim | SwiGLU Hidden | Target Scale |
|---|---:|---:|---:|---:|---:|---:|
| A | 4 | 256 | 4 | 64 | 704 | ~7M |
| B | 6 | 384 | 6 | 64 | 1,024 | ~17M |
| C | 8 | 512 | 8 | 64 | 1,360 | ~34M |

### Core architecture

Each model uses:

- learned token embeddings
- causal multi-head self-attention
- Rotary Position Embeddings (RoPE)
- RMSNorm
- SwiGLU feed-forward networks
- pre-normalization residual blocks
- final RMSNorm
- tied token-embedding / output-projection weights

The implementation is written explicitly in PyTorch rather than using a prebuilt Transformer model.

### Fixed model-level controls

- Vocabulary size: 16,384
- Context length: 512 tokens
- Dropout: 0.10
- Attention head dimension: 64
- Linear projection biases: disabled
- Input/output embedding weights: tied


In [1]:
from dataclasses import dataclass

VOCAB_SIZE = 16_384
CONTEXT_LENGTH = 512
DROPOUT = 0.10


@dataclass(frozen=True)
class ModelConfig:
    name: str
    vocab_size: int
    context_length: int
    n_layers: int
    d_model: int
    n_heads: int
    d_ff: int
    dropout: float = DROPOUT

    @property
    def head_dim(self) -> int:
        return self.d_model // self.n_heads


MODEL_CONFIGS = {
    "A": ModelConfig(
        name="Model A",
        vocab_size=VOCAB_SIZE,
        context_length=CONTEXT_LENGTH,
        n_layers=4,
        d_model=256,
        n_heads=4,
        d_ff=704,
    ),
    "B": ModelConfig(
        name="Model B",
        vocab_size=VOCAB_SIZE,
        context_length=CONTEXT_LENGTH,
        n_layers=6,
        d_model=384,
        n_heads=6,
        d_ff=1024,
    ),
    "C": ModelConfig(
        name="Model C",
        vocab_size=VOCAB_SIZE,
        context_length=CONTEXT_LENGTH,
        n_layers=8,
        d_model=512,
        n_heads=8,
        d_ff=1360,
    ),
}

MODEL_CONFIGS


{'A': ModelConfig(name='Model A', vocab_size=16384, context_length=512, n_layers=4, d_model=256, n_heads=4, d_ff=704, dropout=0.1),
 'B': ModelConfig(name='Model B', vocab_size=16384, context_length=512, n_layers=6, d_model=384, n_heads=6, d_ff=1024, dropout=0.1),
 'C': ModelConfig(name='Model C', vocab_size=16384, context_length=512, n_layers=8, d_model=512, n_heads=8, d_ff=1360, dropout=0.1)}

In [2]:
for key, cfg in MODEL_CONFIGS.items():
    assert cfg.d_model % cfg.n_heads == 0
    assert cfg.head_dim == 64
    assert cfg.context_length == CONTEXT_LENGTH
    assert cfg.vocab_size == VOCAB_SIZE

    print(
        f"{cfg.name}: "
        f"{cfg.n_layers} layers, "
        f"d_model={cfg.d_model}, "
        f"{cfg.n_heads} heads × {cfg.head_dim} dims, "
        f"d_ff={cfg.d_ff}"
    )


Model A: 4 layers, d_model=256, 4 heads × 64 dims, d_ff=704
Model B: 6 layers, d_model=384, 6 heads × 64 dims, d_ff=1024
Model C: 8 layers, d_model=512, 8 heads × 64 dims, d_ff=1360


In [3]:
def analytical_parameter_count(cfg: ModelConfig) -> dict:
    embeddings = cfg.vocab_size * cfg.d_model
    attention_per_layer = 4 * cfg.d_model**2
    swiglu_per_layer = 3 * cfg.d_model * cfg.d_ff
    norms_per_layer = 2 * cfg.d_model
    block_per_layer = attention_per_layer + swiglu_per_layer + norms_per_layer
    transformer_blocks = cfg.n_layers * block_per_layer
    final_norm = cfg.d_model
    total = embeddings + transformer_blocks + final_norm
    return {"embeddings": embeddings, "attention_per_layer": attention_per_layer, "swiglu_per_layer": swiglu_per_layer, "norms_per_layer": norms_per_layer, "block_per_layer": block_per_layer, "transformer_blocks": transformer_blocks, "final_norm": final_norm, "total": total}

for key, cfg in MODEL_CONFIGS.items():
    counts = analytical_parameter_count(cfg)
    print(f"{cfg.name}: {counts['total']:,} parameters ({counts['total'] / 1e6:.2f}M)")


Model A: 7,407,872 parameters (7.41M)
Model B: 16,913,280 parameters (16.91M)
Model C: 33,497,600 parameters (33.50M)


## Parameter-count mental model

For each Transformer block:

- Attention contributes `4 * d_model^2` parameters for Q, K, V, and output projections.
- SwiGLU contributes `3 * d_model * d_ff` parameters for gate, up, and down projections.
- Two RMSNorms contribute `2 * d_model` learned scale parameters.

The token embedding matrix contributes `vocab_size * d_model` parameters and is tied to the language-model output projection.

RoPE adds no learned parameters.

For Model A, the embedding matrix alone contains 4,194,304 parameters, which is about 57% of the full 7.41M-parameter model. This is why vocabulary size and weight tying matter materially at small model scales.


## Chunk 2 — RMSNorm

Before implementing attention or the feed-forward network, we implement the normalization used throughout the model.

### Why normalization is needed

As activations move through many residual blocks, their scale can drift. Normalization keeps the numerical scale of those activations controlled, which generally makes optimization more stable.

A classic **LayerNorm** normalizes using both the mean and variance of the hidden features. RMSNorm is simpler: it rescales the vector using its root-mean-square magnitude without subtracting the mean.

Our architecture uses two RMSNorms inside every Transformer block and one final RMSNorm after the last block.


In [4]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_dtype = x.dtype
        x_float = x.float()
        mean_square = x_float.pow(2).mean(dim=-1, keepdim=True)
        x_normalized = x_float * torch.rsqrt(mean_square + self.eps)
        return (x_normalized * self.weight).to(dtype=input_dtype)


In [5]:
torch.manual_seed(42)
cfg = MODEL_CONFIGS["A"]
norm = RMSNorm(cfg.d_model)
x = torch.randn(2, 5, cfg.d_model)
y = norm(x)
assert y.shape == x.shape
parameter_count = sum(p.numel() for p in norm.parameters())
assert parameter_count == cfg.d_model
output_rms = y.float().pow(2).mean(dim=-1).sqrt()
max_deviation_from_one = (output_rms - 1.0).abs().max().item()
assert max_deviation_from_one < 1e-5
print(f"Input shape:              {tuple(x.shape)}")
print(f"Output shape:             {tuple(y.shape)}")
print(f"RMSNorm parameters:       {parameter_count:,}")
print(f"Expected parameters:      {cfg.d_model:,}")
print(f"Max RMS deviation from 1: {max_deviation_from_one:.2e}")


Input shape:              (2, 5, 256)
Output shape:             (2, 5, 256)
RMSNorm parameters:       256
Expected parameters:      256
Max RMS deviation from 1: 5.96e-07


In [6]:
for key, cfg in MODEL_CONFIGS.items():
    test_norm = RMSNorm(cfg.d_model)
    observed = sum(p.numel() for p in test_norm.parameters())
    expected = cfg.d_model
    assert observed == expected
    print(f"{cfg.name}: RMSNorm = {observed:,} learned parameters")


Model A: RMSNorm = 256 learned parameters
Model B: RMSNorm = 384 learned parameters
Model C: RMSNorm = 512 learned parameters


### RMSNorm mental model
> **RMSNorm controls the magnitude of the hidden-state vector without recentering it.**


## Chunk 3 — Rotary Position Embeddings (RoPE)

RoPE injects positional information by rotating attention query and key vectors as a deterministic function of position. RoPE adds no learned parameters.


In [7]:
class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim: int, max_seq_len: int, base: float = 10_000.0):
        super().__init__()
        if head_dim % 2 != 0:
            raise ValueError("RoPE requires an even head dimension.")
        self.head_dim = head_dim
        self.max_seq_len = max_seq_len
        pair_dims = torch.arange(0, head_dim, 2, dtype=torch.float32)
        inv_freq = base ** (-pair_dims / head_dim)
        positions = torch.arange(max_seq_len, dtype=torch.float32)
        angles = torch.outer(positions, inv_freq)
        self.register_buffer("cos_cached", angles.cos(), persistent=False)
        self.register_buffer("sin_cached", angles.sin(), persistent=False)
    @staticmethod
    def _apply_rotation(x, cos, sin):
        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]
        return torch.stack((x_even * cos - x_odd * sin, x_even * sin + x_odd * cos), dim=-1).flatten(-2)
    def forward(self, q, k):
        seq_len = q.size(-2)
        cos = self.cos_cached[:seq_len].to(device=q.device, dtype=q.dtype)[None, None, :, :]
        sin = self.sin_cached[:seq_len].to(device=q.device, dtype=q.dtype)[None, None, :, :]
        return self._apply_rotation(q, cos, sin), self._apply_rotation(k, cos, sin)


In [8]:
torch.manual_seed(42)
cfg = MODEL_CONFIGS["A"]
rope = RotaryEmbedding(cfg.head_dim, cfg.context_length)
q = torch.randn(2, cfg.n_heads, 16, cfg.head_dim)
k = torch.randn(2, cfg.n_heads, 16, cfg.head_dim)
q_rot, k_rot = rope(q, k)
assert q_rot.shape == q.shape and k_rot.shape == k.shape
assert sum(p.numel() for p in rope.parameters()) == 0
zero_position_error = (q_rot[:, :, 0] - q[:, :, 0]).abs().max().item()
norm_error = (q_rot.float().norm(dim=-1)-q.float().norm(dim=-1)).abs().max().item()
print(f"Q shape:                    {tuple(q.shape)}")
print(f"Rotated Q shape:            {tuple(q_rot.shape)}")
print("RoPE learned parameters:    0")
print(f"Position-0 identity error:  {zero_position_error:.2e}")
print(f"Max norm-preservation error: {norm_error:.2e}")


Q shape:                    (2, 4, 16, 64)
Rotated Q shape:            (2, 4, 16, 64)
RoPE learned parameters:    0
Position-0 identity error:  0.00e+00
Max norm-preservation error: 9.54e-07


## Chunk 4 — Causal Multi-Head Self-Attention

Attention routes information between tokens. Q and K determine where to look, V determines what information is retrieved, and the causal mask prevents future-token leakage.


In [12]:
import math
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.d_model, self.n_heads, self.head_dim = cfg.d_model, cfg.n_heads, cfg.head_dim
        self.max_seq_len = cfg.context_length
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, self.max_seq_len)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        causal_mask = torch.tril(torch.ones(self.max_seq_len, self.max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask[None,None,:,:], persistent=False)
    def _split_heads(self,x):
        b,t,_=x.shape
        return x.view(b,t,self.n_heads,self.head_dim).transpose(1,2)
    def _merge_heads(self,x):
        b,_,t,_=x.shape
        return x.transpose(1,2).contiguous().view(b,t,self.d_model)
    def forward(self,x,return_attention=False):
        _,seq_len,_=x.shape
        q=self._split_heads(self.q_proj(x)); k=self._split_heads(self.k_proj(x)); v=self._split_heads(self.v_proj(x))
        q,k=self.rope(q,k)
        scores=torch.matmul(q,k.transpose(-2,-1))/math.sqrt(self.head_dim)
        mask=self.causal_mask[:,:,:seq_len,:seq_len]
        scores=scores.masked_fill(~mask,float('-inf'))
        attention=torch.softmax(scores.float(),dim=-1).to(dtype=v.dtype)
        attention=self.attn_dropout(attention)
        context=torch.matmul(attention,v)
        output=self.resid_dropout(self.out_proj(self._merge_heads(context)))
        return (output,attention) if return_attention else output


In [13]:
torch.manual_seed(42)
cfg=MODEL_CONFIGS["A"]
attn=CausalSelfAttention(cfg); attn.eval()
x=torch.randn(2,16,cfg.d_model)
with torch.no_grad(): y,weights=attn(x,return_attention=True)
observed_parameters=sum(p.numel() for p in attn.parameters())
expected_parameters=4*cfg.d_model**2
assert y.shape==x.shape and observed_parameters==expected_parameters
print(f"Input shape:                 {tuple(x.shape)}")
print(f"Output shape:                {tuple(y.shape)}")
print(f"Attention-weight shape:      {tuple(weights.shape)}")
print(f"Observed attention params:   {observed_parameters:,}")
print(f"Expected attention params:   {expected_parameters:,}")


Input shape:                 (2, 16, 256)
Output shape:                (2, 16, 256)
Attention-weight shape:      (2, 4, 16, 16)
Observed attention params:   262,144
Expected attention params:   262,144


In [14]:
row_sums=weights.float().sum(dim=-1)
row_sum_error=(row_sums-1.0).abs().max().item()
future_mask=torch.triu(torch.ones(16,16,dtype=torch.bool),diagonal=1)
max_future_attention=weights[...,future_mask].abs().max().item()
assert row_sum_error<1e-6 and max_future_attention==0.0
print(f"Max attention-row sum error: {row_sum_error:.2e}")
print(f"Max future attention weight: {max_future_attention:.2e}")


Max attention-row sum error: 1.79e-07
Max future attention weight: 0.00e+00


In [15]:
torch.manual_seed(42)
test_x=torch.randn(1,8,cfg.d_model); changed_x=test_x.clone(); changed_x[:,7,:]=torch.randn_like(changed_x[:,7,:])*100.0
with torch.no_grad(): original_output=attn(test_x); changed_output=attn(changed_x)
earlier_difference=(original_output[:,:7]-changed_output[:,:7]).abs().max().item()
final_position_difference=(original_output[:,7]-changed_output[:,7]).abs().max().item()
assert earlier_difference<1e-6 and final_position_difference>1e-3
print(f"Max difference at positions 0..6: {earlier_difference:.2e}")
print(f"Difference at changed position 7:  {final_position_difference:.2e}")


Max difference at positions 0..6: 0.00e+00
Difference at changed position 7:  9.08e+01


## Chunk 5 — SwiGLU Feed-Forward Network

Attention mixes information across token positions. SwiGLU transforms each token independently using a learned gating branch and candidate-feature branch.


In [16]:
import torch.nn.functional as F
class SwiGLU(nn.Module):
    def __init__(self,cfg:ModelConfig):
        super().__init__()
        self.d_model,self.d_ff=cfg.d_model,cfg.d_ff
        self.gate_proj=nn.Linear(cfg.d_model,cfg.d_ff,bias=False)
        self.up_proj=nn.Linear(cfg.d_model,cfg.d_ff,bias=False)
        self.down_proj=nn.Linear(cfg.d_ff,cfg.d_model,bias=False)
        self.dropout=nn.Dropout(cfg.dropout)
    def forward(self,x):
        gate=F.silu(self.gate_proj(x))
        up=self.up_proj(x)
        return self.dropout(self.down_proj(gate*up))


In [17]:
torch.manual_seed(42)
cfg=MODEL_CONFIGS["A"]
ffn=SwiGLU(cfg); ffn.eval()
x=torch.randn(2,16,cfg.d_model)
with torch.no_grad(): y=ffn(x)
observed_parameters=sum(p.numel() for p in ffn.parameters())
expected_parameters=3*cfg.d_model*cfg.d_ff
assert y.shape==x.shape and observed_parameters==expected_parameters
print(f"Input shape:               {tuple(x.shape)}")
print(f"Output shape:              {tuple(y.shape)}")
print(f"Model A d_ff:              {cfg.d_ff:,}")
print(f"Observed SwiGLU params:    {observed_parameters:,}")
print(f"Expected SwiGLU params:    {expected_parameters:,}")


Input shape:               (2, 16, 256)
Output shape:              (2, 16, 256)
Model A d_ff:              704
Observed SwiGLU params:    540,672
Expected SwiGLU params:    540,672


In [18]:
torch.manual_seed(42)
test_x=torch.randn(1,8,cfg.d_model); changed_x=test_x.clone(); changed_x[:,4,:]=torch.randn_like(changed_x[:,4,:])*100.0
with torch.no_grad(): original_output=ffn(test_x); changed_output=ffn(changed_x)
unchanged_positions=[0,1,2,3,5,6,7]
other_token_difference=(original_output[:,unchanged_positions]-changed_output[:,unchanged_positions]).abs().max().item()
changed_token_difference=(original_output[:,4]-changed_output[:,4]).abs().max().item()
assert other_token_difference==0.0 and changed_token_difference>1e-3
print(f"Max difference at unchanged tokens: {other_token_difference:.2e}")
print(f"Difference at changed token 4:       {changed_token_difference:.2e}")


Max difference at unchanged tokens: 0.00e+00
Difference at changed token 4:       2.90e+03


In [19]:
probe=torch.randn(1,3,cfg.d_model)
with torch.no_grad():
    gate_values=F.silu(ffn.gate_proj(probe)); up_values=ffn.up_proj(probe); gated_hidden=gate_values*up_values; projected_back=ffn.down_proj(gated_hidden)
print(f"Input:          {tuple(probe.shape)}")
print(f"Gate branch:    {tuple(gate_values.shape)}")
print(f"Up branch:      {tuple(up_values.shape)}")
print(f"Gated hidden:   {tuple(gated_hidden.shape)}")
print(f"Projected back: {tuple(projected_back.shape)}")


Input:          (1, 3, 256)
Gate branch:    (1, 3, 704)
Up branch:      (1, 3, 704)
Gated hidden:   (1, 3, 704)
Projected back: (1, 3, 256)


In [20]:
for key,model_cfg in MODEL_CONFIGS.items():
    test_ffn=SwiGLU(model_cfg)
    observed=sum(p.numel() for p in test_ffn.parameters())
    expected=3*model_cfg.d_model*model_cfg.d_ff
    assert observed==expected
    print(f"{model_cfg.name}: d_model={model_cfg.d_model}, d_ff={model_cfg.d_ff}, SwiGLU parameters={observed:,}")


Model A: d_model=256, d_ff=704, SwiGLU parameters=540,672
Model B: d_model=384, d_ff=1024, SwiGLU parameters=1,179,648
Model C: d_model=512, d_ff=1360, SwiGLU parameters=2,088,960


### SwiGLU mental model
> **Attention lets a token gather information from other tokens. SwiGLU then transforms that gathered information inside the token's own feature vector.**
